In [ ]:
import os
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, TextVectorization

# First dataset
print("Downloading first dataset (spam-email-dataset)...")
path1 = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")
csv_files1 = [f for f in os.listdir(path1) if f.endswith('.csv')]
csv_path1 = os.path.join(path1, csv_files1[0])
df1 = pd.read_csv(csv_path1)

# Standardize column names
df1 = df1.rename(columns={'text': 'text', 'spam': 'spam'})
df1 = df1[['text', 'spam']]
print(f"Dataset 1 shape: {df1.shape}")

# second dataset
print("\nDownloading second dataset (email-spam-classification-dataset)...")
path2 = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
csv_files2 = [f for f in os.listdir(path2) if f.endswith('.csv')]
csv_path2 = os.path.join(path2, csv_files2[0])
df2 = pd.read_csv(csv_path2)

# Identify columns dynamically
text_col_df2 = 'email' if 'email' in df2.columns else df2.columns[1]
label_col_df2 = 'label' if 'label' in df2.columns else df2.columns[0]

df2_cleaned = pd.DataFrame()
df2_cleaned['text'] = df2[text_col_df2].astype(str)

# Standardize string labels ('spam' / 'ham') to binary numeric values
if df2[label_col_df2].dtype == object:
    df2_cleaned['spam'] = df2[label_col_df2].str.lower().map({'spam': 1, 'ham': 0})
else:
    df2_cleaned['spam'] = df2[label_col_df2].astype(int)

df2_cleaned = df2_cleaned.dropna().reset_index(drop=True)
print(f"Dataset 2 standardized shape: {df2_cleaned.shape}")

# merge datasets, shuffle them, separate into trainging and testing sets
combined_df = pd.concat([df1, df2_cleaned], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nCombined Dataset shape: {combined_df.shape}")
print(combined_df['spam'].value_counts())

X_comb = combined_df['text'].astype(str).values
y_comb = combined_df['spam'].values

# 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_comb, y_comb, test_size=0.30, random_state=42, stratify=y_comb
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nFinal Splits:")
print(f"- Train size: {len(X_train)}")
print(f"- Validation (Preliminary) size: {len(X_val)}")
print(f"- Testing size: {len(X_test)}")

# Model training
max_features = 12000
sequence_length = 250

vectorize_layer = TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Train vectorize layer on the training set
vectorize_layer.adapt(X_train)

model = Sequential([
    vectorize_layer,
    Embedding(max_features, 64),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining the model on the merged dataset...")
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val)
)

# test sets
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Model Performance on Final Testing Set ---")
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")

Using Colab cache for faster access to the 'spam-email-dataset' dataset.
Dataset 1 shape: (5728, 2)

Using Colab cache for faster access to the 'email-spam-classification-dataset' dataset.
Dataset 2 standardized shape: (83448, 2)

Combined Dataset shape: (89176, 2)
spam
1    45278
0    43898
Name: count, dtype: int64

Final Splits:
- Train size: 62423
- Validation (Preliminary) size: 13376
- Testing size: 13377

Training the model on the merged dataset...
Epoch 1/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - accuracy: 0.9391 - loss: 0.1575 - val_accuracy: 0.9780 - val_loss: 0.0679
Epoch 2/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.9778 - loss: 0.0613 - val_accuracy: 0.9810 - val_loss: 0.0560
Epoch 3/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9821 - loss: 0.0496 - val_accuracy: 0.9821 - val_loss: 0.0512
Epoch 4/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.9854 - loss: 0.0435 - val_accuracy: 0.9867 - val_loss: 0.0477
Epoch 5/5
1951/1951 

In [ ]:
#@title Interactive Spam Classifier
# Run this cell to test custom email transcripts against the trained Keras model.

import numpy as np

def predict_email(email_text):
    # Convert the raw text into an object array so Keras can process the string correctly
    input_data = np.array([email_text], dtype=object)
    prediction = improved_model.predict(input_data, verbose=0)[0][0]

    print("\n" + "="*40)
    print("RESULTS:")
    print("="*40)
    if prediction >= 0.9:
        print(f"Prediction: SPAM (Confidence: {prediction * 100:.2f}%)")
    else:
        print(f"Prediction: NOT SPAM (Confidence: {(1 - prediction) * 100:.2f}%)")
    print("="*40)

# Prompt user for input
user_email = input("Paste the email transcript here to analyze: ")
if user_email.strip():
    predict_email(user_email)
else:
    print("Please provide a valid text input next time!")

Paste the email transcript here to analyze: Subject: Get 5.00% APY on both Checking and Savings  Hi Micah, Ready to earn higher yield on both your Checking and Savings accounts? You can earn 5.00% APY for 90 days once you set up and receive a qualifying direct deposit.¹ And after 90 days, you’ll still earn up to 3.60% APY.² This is what better banking looks like. Open your accounts today to start putting your money to work.

RESULTS:
Prediction: SPAM (Confidence: 99.73%)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.layers import TextVectorization

# 1. Use TF-IDF representation with up to 2-grams (bigrams) for better context capturing
max_features = 20000

tf_idf_vectorizer = TextVectorization(
    max_tokens=max_features,
    output_mode='tf-idf',
    ngrams=(1, 2)
)

print("Adapting TF-IDF Vectorizer on training data...")
tf_idf_vectorizer.adapt(X_train)

# 2. Build a deeper Multi-Layer Perceptron (MLP) model suited for TF-IDF vectors
improved_model = Sequential([
    Input(shape=(1,), dtype=tf.string),
    tf_idf_vectorizer,
    Dense(64, activation='relu'),
    Dropout(0.4),  # Stronger regularization
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

improved_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining the improved TF-IDF model...")
history_improved = improved_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,  # Increased batch size for smoother convergence
    validation_data=(X_val, y_val)
)

# 3. Evaluate performance on the test set
imp_loss, imp_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Improved Model Performance on Testing Set ---")
print(f"Test Loss: {imp_loss:.4f} (Old: {loss:.4f})")
print(f"Test Accuracy: {imp_accuracy*100:.2f}% (Old: {accuracy*100:.2f}%)")

Adapting TF-IDF Vectorizer on training data...

Training the improved TF-IDF model...
Epoch 1/5
976/976 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - accuracy: 0.9588 - loss: 0.1892 - val_accuracy: 0.9892 - val_loss: 0.0617
Epoch 2/5
976/976 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step - accuracy: 0.9845 - loss: 0.0720 - val_accuracy: 0.9885 - val_loss: 0.0716
Epoch 3/5
976/976 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step - accuracy: 0.9896 - loss: 0.0491 - val_accuracy: 0.9899 - val_loss: 0.0500
Epoch 4/5
976/976 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.9909 - loss: 0.0426 - val_accuracy: 0.9904 - val_loss: 0.0487
Epoch 5/5
976/976 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.9936 - loss: 0.0299 - val_accuracy: 0.9911 - val_loss: 0.0402

--- Improved Model Performance on Testing Set ---
Test Loss: 0.0372 (Old: 0.0557)
Test Accuracy: 99.10% (Old: 98.21%)
